# Google News AI Scraper (Simple Stable Version - V2)

This notebook scrapes AI news from Google News RSS feed within a specified date range.

In [17]:
# =============================================================================
# GOOGLE NEWS SCRAPER [VERSION 12.0 - DIRECT LINK EXTRACTION]
# =============================================================================

import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import time
import re

HAS_CURL_CFFI = False 

def parse_rss_date(date_str):
    if not date_str: return None
    try:
        dt = datetime.strptime(date_str, "%a, %d %b %Y %H:%M:%S %Z")
        return dt
    except ValueError:
        pass
    try:
        dt = datetime.strptime(date_str.rsplit(' ', 1)[0], "%a, %d %b %Y %H:%M:%S")
        return dt
    except ValueError:
        pass
    return None

def resolve_url(url):
    """Resolves Google News redirects aggressively. (BYPASSED IN V12)"""
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
        resp = requests.get(url, headers=headers, allow_redirects=True, timeout=10)
        final_url = resp.url
        if "news.google.com" not in final_url and "consent.google.com" not in final_url:
            return final_url    
        return final_url
    except Exception as e:
        return url

def is_code_like(text):
    """Returns True if text looks like JS/CSS code."""
    # Heuristic: High density of code symbols or specific keywords
    if "function()" in text or "function ()" in text: return True
    if "var " in text and "=" in text: return True
    if "{" in text and "}" in text and ";" in text: return True
    if "@media" in text or "@font-face" in text: return True
    
    # High symbol density check
    symbols = text.count('{') + text.count('}') + text.count(';') + text.count('(') + text.count(')')
    if len(text) > 0 and (symbols / len(text)) > 0.05:
        return True
    return False

def extract_article_content(url):
    print(f"   [EXTRACT - V12] {url[:60]}...")
    try:
        # V12 CHANGE: ALLOW Google News URLs (Hope requests follows redirect)
        if "google-analytics" in url or "angular" in url:
             print("   -> SKIP: Bad URL (Tech)")
             return ""
            
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
        try:
            # allow_redirects=True is default, ensuring we follow the Google Redirect
            response = requests.get(url, headers=headers, timeout=15, allow_redirects=True)
        except:
            return ""
            
        html = response.text
        
        # --- REGEX STRATEGY ---
        # pattern: <p (anything) > (content) </p>
        pattern = r'<p[^>]*>(.*?)</p>'
        matches = re.findall(pattern, html, re.DOTALL | re.IGNORECASE)
        
        print(f"   -> Found {len(matches)} raw regex matches.")

        valid_paragraphs = []
        
        for m in matches:
            # Clean tags
            text = re.sub(r'<[^>]+>', '', m).strip()
            
            if len(text) > 40 and not is_code_like(text) and "cookie" not in text.lower():
                valid_paragraphs.append(text)
        
        if not valid_paragraphs:
             print("   -> No valid paragraphs found after regex filter.")
             return ""
             
        count = min(len(valid_paragraphs), 10)
        print(f"   -> Returning top {count} paragraphs.")
        final_text = "\n\n".join(valid_paragraphs[:count])
        return final_text
    except Exception as e:
        print(f"Fatal Extract Error: {e}")
        return ""

def get_google_news_ai(start_date_str, end_date_str, limit=None):
    rss_url = "https://news.google.com/rss/search?q=AI&hl=en-US&gl=US&ceid=US:en"
    
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    end_date = end_date.replace(hour=23, minute=59, second=59)

    try:
        response = requests.get(rss_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        if response.status_code != 200: return []
        soup = BeautifulSoup(response.content, 'xml')
        items = soup.find_all('item')
        
        print(f"--- STARTING PROCESS (VERSION 12.0 - DIRECT LINK) ---")
        print(f"Found {len(items)} items. Limit set to: {limit}")
        
        articles = []
        processed_count = 0
        
        for item in items:
            if limit and processed_count >= limit:
                print(f"*** LIMIT REACHED ({limit}) - STOPPING ***")
                break
            
            try:
                title = item.title.get_text(strip=True)
                pub_date_str = item.pubDate.get_text(strip=True)
                source_name = item.source.get_text(strip=True) if item.source else "Unknown"
                
                # 1. DIRECT USE OF <LINK>
                link = item.link.get_text(strip=True)
                final_url = link
                
                date_obj = parse_rss_date(pub_date_str)
                if not date_obj: continue
                
                date_obj_naive = date_obj.replace(tzinfo=None)
                if not (start_date <= date_obj_naive <= end_date): continue

                processed_count += 1
                print(f"[{processed_count}/{limit}] Processing: {title[:40]}...")

                # 2. Extract directly (requests will auto-follow 302)
                raw_content = extract_article_content(final_url)
                
                article_data = {
                    'raw_content': raw_content,
                    'source_url': final_url,
                    'date': date_obj.strftime("%b %d, %Y"),
                    'raw_title': title,
                    'source_name': source_name
                }
                articles.append(article_data)
                time.sleep(1)
                
            except Exception as e:
                print(f"Error on item: {e}")
                continue
                
        return articles
    except Exception as e:
        print(f"Fatal error: {e}")
        return []

# =========================================
# EXECUTION
# =========================================
if __name__ == "__main__":
    s = "2026-01-04"
    e = "2026-01-05"
    res = get_google_news_ai(s, e, limit=3)
    print(f"\nCompleted. Extracted {len(res)} articles.")
    if res:
        print("\n--- First Article Content Sample ---")
        print(res[0]['raw_content'][:500])


--- STARTING PROCESS (VERSION 12.0 - DIRECT LINK) ---
Found 102 items. Limit set to: 3
[1/3] Processing: World ‘may not have time’ to prepare for...
   [EXTRACT - V12] https://news.google.com/rss/articles/CBMiyAFBVV95cUxNVUVLUVl...
   -> Found 0 raw regex matches.
   -> No valid paragraphs found after regex filter.
[2/3] Processing: Is the AI boom a bubble waiting to pop? ...
   [EXTRACT - V12] https://news.google.com/rss/articles/CBMifEFVX3lxTE9rdC1vc2x...
   -> Found 0 raw regex matches.
   -> No valid paragraphs found after regex filter.
[3/3] Processing: If AI doesn't produce a bubble it 'will ...
   [EXTRACT - V12] https://news.google.com/rss/articles/CBMiowFBVV95cUxNVldZUGZ...
   -> Found 0 raw regex matches.
   -> No valid paragraphs found after regex filter.
*** LIMIT REACHED (3) - STOPPING ***

Completed. Extracted 3 articles.

--- First Article Content Sample ---

